# Assignment #3 -- Baseline: EDA and Corrected Replication

**Paper.** Najafi-Zangeneh, Shams-Gharneh, Arjomandi-Nezhad & Hashemkhani Zolfani, *An Improved Machine Learning-Based Employees Attrition Prediction Framework with Emphasis on Feature Selection*, Mathematics 2021, 9, 1226.

**Dataset.** IBM HR Analytics Employee Attrition (`data/mainData.csv`), 1470 rows x 35 columns.

**Purpose.** Step 1 of Assignment #3 -- establish the baseline. This notebook reproduces the Assignment #2 *corrected* replication (EDA through model evaluation) unchanged: same preprocessing, features, model configuration, evaluation metrics, split, and procedure. Every subsequent Assignment #3 experiment is compared against the numbers produced here. All computation lives in `src/`; this notebook only imports, calls, displays, and narrates.

## Part 1 -- Exploratory Data Analysis

Each section answers one question that feeds a decision in Part 2. All computation is in `src/eda.py`; figures are built by `src/plotting.py`.

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent / 'src'))
import pandas as pd
pd.set_option('display.max_columns', 60)
pd.set_option('display.width', 170)
import eda
from data import load_raw
df = load_raw()
tables = eda.run_all(df)
df.head()

EDA: 9 tables -> outputs/tables/eda_*.csv, 5 figures -> outputs/figures/0[1-5]_*.png


,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EmployeeCount,EmployeeNumber,EnvironmentSatisfaction,Gender,HourlyRate,JobInvolvement,JobLevel,JobRole,JobSatisfaction,MaritalStatus,MonthlyIncome,MonthlyRate,NumCompaniesWorked,Over18,OverTime,PercentSalaryHike,PerformanceRating,RelationshipSatisfaction,StandardHours,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,41,Yes,Travel_Rarely,1102,Sales,1,2,Life Sciences,1,1,2,Female,94,3,2,Sales Executive,4,Single,5993,19479,8,Y,Yes,11,3,1,80,0,8,0,1,6,4,0,5
1,49,No,Travel_Frequently,279,Research & Development,8,1,Life Sciences,1,2,3,Male,61,2,2,Research Scientist,2,Married,5130,24907,1,Y,No,23,4,4,80,1,10,3,3,10,7,1,7
2,37,Yes,Travel_Rarely,1373,Research & Development,2,2,Other,1,4,4,Male,92,2,1,Laboratory Technician,3,Single,2090,2396,6,Y,Yes,15,3,2,80,0,7,3,3,0,0,0,0
3,33,No,Travel_Frequently,1392,Research & Development,3,4,Life Sciences,1,5,4,Female,56,3,1,Research Scientist,3,Married,2909,23159,1,Y,Yes,11,3,3,80,0,8,3,3,8,7,3,0
4,27,No,Travel_Rarely,591,Research & Development,2,1,Medical,1,7,1,Male,40,3,1,Laboratory Technician,2,Married,3468,16632,9,Y,No,12,3,4,80,1,6,3,3,2,2,2,2


### 1.1 Dataset verification against the paper

Observation count, feature count and types, target, missing values, duplicates, class distribution - and every difference from the paper's description.

In [2]:
tables['verification']

,property,dataset,paper,note
0,Observations,1470,1470,match
1,Columns (incl. target),35,35,match
2,Feature count,34,34 listed,"paper drops 2 constants -> 32, and counts Empl..."
3,Constant columns,"EmployeeCount, Over18, StandardHours","StandardHours, EmployeeCount",Over18 is also constant; the paper never names it
4,Missing cells,0,not discussed,none -> no imputation step
5,Exact duplicate rows,0,not discussed,none
6,Duplicates ignoring EmployeeNumber,0,not discussed,none -> rows are distinct employees
7,Target,Attrition,Attrition,match
8,Class distribution,"{'No': 1233, 'Yes': 237}",imbalanced,16.1% positive (Yes)


**Findings.**

- Row and column counts match the paper exactly (1470 x 35).
- **Three** columns are constant, not two: the paper names `StandardHours` and `EmployeeCount` but never mentions `Over18` (constant `'Y'`). We drop the two the paper drops and let feature selection remove the `Over18` dummy, as the paper implicitly does.
- The paper lists `EmployeeNumber` among the predictive features; it is a pure row identifier, so we drop it (31 modelling features vs the paper's 32).
- **No missing cells and no duplicate rows**, even after ignoring `EmployeeNumber` - so no imputation or de-duplication step is needed.
- `Attrition` is binary and imbalanced.

### 1.2 Class imbalance

The single most consequential property of the dataset for both modelling and evaluation.

In [3]:
tables['class_counts']

,class,count
0,No,1233
1,Yes,237


![class balance](../outputs/figures/01_class_balance.png)

**Finding.** Only **16.1%** of employees left (237 of 1470). A model that always predicts "stays" scores 83.9% accuracy while being useless, so accuracy cannot be the headline metric - the paper argues the same and reports precision / recall / F1. This also motivates `class_weight='balanced'` in Part 2: without it the minority-class recall collapses and the paper's reported recall of 0.82 is unreachable.

### 1.3 Relationship between features and the target

In [4]:
tables['numeric_association']

,feature,mean_stay,mean_leave,std_mean_diff
0,JobLevel,2.15,1.64,-0.493
1,TotalWorkingYears,11.86,8.24,-0.484
2,MonthlyIncome,6832.74,4787.09,-0.479
3,YearsInCurrentRole,4.48,2.90,-0.462
4,YearsWithCurrManager,4.37,2.85,-0.449
5,Age,37.56,33.61,-0.425
6,StockOptionLevel,0.85,0.53,-0.374
7,YearsAtCompany,7.37,5.13,-0.372
8,JobInvolvement,2.77,2.52,-0.343
9,JobSatisfaction,2.78,2.47,-0.281


![numeric association](../outputs/figures/02_numeric_target_association.png)

In [5]:
tables['categorical_association']

,feature,levels,min_rate,max_rate,spread
0,JobRole,9,0.025,0.398,0.373
1,OverTime,2,0.104,0.305,0.201
2,BusinessTravel,3,0.080,0.249,0.169
3,MaritalStatus,3,0.101,0.255,0.154
4,EducationField,6,0.134,0.259,0.125
5,Department,3,0.138,0.206,0.068
6,Gender,2,0.148,0.170,0.022


![categorical association](../outputs/figures/03_categorical_target_association.png)

**Findings.** Leavers are younger, have fewer total working years, shorter tenure, lower job level and income, and live farther from the office (all |Cohen's d| ~ 0.2-0.5). Among categoricals the strongest signal is **`OverTime`** (31% vs 10%), then `JobRole` (Sales Representative ~40%), frequent business travel (25%), and `MaritalStatus` Single (26%). These directions match domain expectation and give a sanity check for Part 2: a feature-selection result that discards `OverTime` or `TotalWorkingYears` would be suspect.

### 1.4 Correlation and redundancy among features

The paper reads logistic-regression coefficients directly as "feature importance". That is only safe if the features are not badly collinear.

In [6]:
tables['high_correlations']

,feature_a,feature_b,r
0,JobLevel,MonthlyIncome,0.950300
1,JobLevel,TotalWorkingYears,0.782208
2,PercentSalaryHike,PerformanceRating,0.773550
3,MonthlyIncome,TotalWorkingYears,0.772893
4,YearsAtCompany,YearsWithCurrManager,0.769212
...,...,...,...
248,NaN,NaN,NaN
249,NaN,NaN,NaN
250,NaN,NaN,NaN
251,NaN,NaN,NaN


In [7]:
tables['vif'].head(12)

,feature,VIF
0,JobLevel,11.205
1,MonthlyIncome,10.800
2,TotalWorkingYears,4.768
3,YearsAtCompany,4.587
4,YearsWithCurrManager,2.775
5,YearsInCurrentRole,2.719
6,PercentSalaryHike,2.516
7,PerformanceRating,2.514
8,Age,2.045
9,YearsSinceLastPromotion,1.674


![correlation heatmap](../outputs/figures/04_correlation_heatmap.png)

**Findings.** `JobLevel` and `MonthlyIncome` are near-duplicates (r = 0.95; VIF ~ 11 each). `TotalWorkingYears` correlates with both, and the four `Years*` tenure columns form a second correlated block (r = 0.6-0.8). **Implication for Part 2:** individual logistic-regression coefficients in this space are unstable - a small change in the training sample can shift weight between `JobLevel` and `MonthlyIncome` - so the paper's coefficient-as-importance narrative needs caution, and the bootstrap coefficient-stability analysis (paper Section 5.3) is exactly the right diagnostic to keep. This is also the strongest candidate for an Assignment #3 feature-redundancy experiment.

### 1.5 Are the ordinal survey scales really numeric?

The paper feeds 1-4 / 1-5 satisfaction and involvement ratings to the model as continuous numbers. We inherit that; here is what is being assumed.

In [8]:
tables['ordinal_scales']

,feature,distinct_values,range,value_counts
0,Education,5,1-5,"{1: 170, 2: 282, 3: 572, 4: 398, 5: 48}"
1,EnvironmentSatisfaction,4,1-4,"{1: 284, 2: 287, 3: 453, 4: 446}"
2,JobInvolvement,4,1-4,"{1: 83, 2: 375, 3: 868, 4: 144}"
3,JobLevel,5,1-5,"{1: 543, 2: 534, 3: 218, 4: 106, 5: 69}"
4,JobSatisfaction,4,1-4,"{1: 289, 2: 280, 3: 442, 4: 459}"
5,PerformanceRating,2,3-4,"{3: 1244, 4: 226}"
6,RelationshipSatisfaction,4,1-4,"{1: 276, 2: 303, 3: 459, 4: 432}"
7,StockOptionLevel,4,0-3,"{0: 631, 1: 596, 2: 158, 3: 85}"
8,WorkLifeBalance,4,1-4,"{1: 80, 2: 344, 3: 893, 4: 153}"


**Finding.** Nine features are discrete ratings with 2-5 levels. Treating them as numeric assumes equal spacing between levels - a documented assumption carried from the paper, not an independent choice. `PerformanceRating` has only two observed values (3, 4) and is effectively binary. Another candidate Assignment #3 experiment: encode these as categorical instead of numeric.

### 1.6 Outliers and anomalous observations

In [9]:
tables['outliers']

,feature,n_outliers,pct,max,upper_fence,skew
0,MonthlyIncome,114,7.8,19999,16581.0,1.37
6,YearsSinceLastPromotion,107,7.3,15,7.5,1.98
4,YearsAtCompany,104,7.1,40,18.0,1.76
3,TotalWorkingYears,63,4.3,40,28.5,1.12
8,NumCompaniesWorked,52,3.5,9,8.5,1.03
5,YearsInCurrentRole,21,1.4,18,14.5,0.92
7,YearsWithCurrManager,14,1.0,17,14.5,0.83
1,MonthlyRate,0,0.0,26999,39083.2,0.02
2,DailyRate,0,0.0,1499,2195.0,-0.00
9,DistanceFromHome,0,0.0,29,32.0,0.96


![outliers](../outputs/figures/05_outliers_strip.png)

**Findings.** The monetary and tenure columns are right-skewed with long upper tails: ~7-8% of `MonthlyIncome`, `YearsAtCompany` and `YearsSinceLastPromotion` sit beyond the 1.5x IQR fence. Split by attrition, the tails are not concentrated among leavers - leavers are simply shifted toward lower values overall. These are plausible values (long-serving senior staff), not data errors, so we keep them and rely on standardisation; the paper applies no outlier treatment and we match that.

### 1.7 Data-leakage audit

In [10]:
tables['leakage']

,item,type,assessment
0,EmployeeNumber,row identifier,no predictive meaning; dropped. Paper keeps it...
1,StandardHours / EmployeeCount,constant,zero information; paper drops before selection...
2,Over18,constant ('Y'),zero information; paper keeps it until selecti...
3,all remaining features,employee attributes / history,measured at or before the observation point --...
4,temporal structure,none,"cross-sectional snapshot, no timestamp -> the ..."


**Findings.** `EmployeeNumber` is an identifier and is dropped; the constant columns carry no information. No feature is measured *after* the attrition outcome, so there is no target leakage inside the feature set. The dataset is a single cross-sectional snapshot with no timestamp, so the brief's time-series checks (trend, seasonality, missing timestamps, autocorrelation) **do not apply**. The real leakage risk here is **procedural**: fitting scaling or feature selection before the train/test split - which is exactly what Part 2 corrects, and which every Assignment #3 experiment must also preserve (fit inside the fold, never before it).

### EDA summary and implications for the baseline

| # | Finding | Implication |
|---|---|---|
| 1 | 1470 x 35, no missing values, no duplicates | Clean start; no imputation / de-duplication |
| 2 | `Over18` is a third constant; paper counts `EmployeeNumber` as a feature | Drop `EmployeeNumber`; let selection remove `Over18` (documented deviations) |
| 3 | Target only 16.1% positive | Report P/R/F1 not accuracy; `class_weight='balanced'` for every model; tune the decision threshold |
| 4 | `OverTime`, tenure, `TotalWorkingYears`, income are the strongest signals | Sanity check for feature selection |
| 5 | `JobLevel` ~ `MonthlyIncome` (r 0.95, VIF ~ 11); `Years*` block correlated | Coefficient-as-importance is fragile; keep the bootstrap stability analysis |
| 6 | 9 ordinal scales treated as continuous | Inherited equal-spacing assumption; `PerformanceRating` near-binary |
| 7 | ~7-8% right-tail outliers, all plausible, not tied to attrition | Keep rows, standardise, no special treatment |
| 8 | Cross-sectional, no time dimension | Time-series checks N/A; leakage fix is procedural - split before preprocessing |


## Part 2 -- Baseline Pipeline: Replicate and Correct the Paper's Methodology

**Paper's pipeline.** data -> dummy-code categoricals -> "max-out" (1-max-out = greedy backward) feature selection -> logistic regression -> accuracy / precision / recall / F1 -> 300-bootstrap coefficient-confidence analysis.

**Paper's reported test results (Section 5.2).** accuracy 0.81, precision 0.43, recall 0.82, F1 0.56 with max-out; 0.78 / 0.39 / 0.82 / 0.53 without it.

Logic lives in `src/`: `data.py` (split-first), `preprocessing.py` (fit on train only), `maxout.py` (leakage-safe feature search), `evaluation.py` (metrics, threshold, bootstrap), `experiment.py` (orchestration), `plotting.py` (figures).

### 2.1 The paper's pipeline, and where it is unclear or wrong

| Stage | What the paper does | Problem | Our correction |
|---|---|---|---|
| Preprocessing scope | Section 5.1 splits train+val / test inside the feature search, but the ordinal scaling and the Section 5.3 bootstrap are described on the whole dataset | Section 5.3 says the bootstrap uses "the original dataset"; Section 3 says the training set - a contradiction. The scaling choice is never stated. | **Split first** (stratified 70/15/15). Fit scaler + encoder on train only. Bootstrap the **training** set. |
| Feature selection | 1-max-out over 7 random train/val splits; drop features removed > 4 times | Sound, but the preprocessor is fit once on all of train+val before the 7 splits, leaking validation statistics into each run | Re-fit the preprocessor on each run's inner-training rows |
| Model comparison | LR + max-out compared against RF / KNN / SVM **taken from other papers** with different splits, preprocessing and metrics (Figure 6) | Not a controlled experiment; it cannot show that LR, or max-out, is the better choice | Run **LR and RF on one identical split / seed / preprocessor**, each with its own max-out search (RF selects with RF as the wrapper) |
| Operating point | recall 0.82 at 16% prevalence reported with no mention of threshold or weighting | Unreproducible; and `class_weight='balanced'` shifts LR's boundary but not RF's majority vote, so a fixed 0.5 cut compares them unfairly | Apply `class_weight='balanced'` to **both**; choose each model's threshold by maximising **validation** F1; also report threshold-free ROC-AUC / PR-AUC |
| Split ratios / seed / regularisation | Not stated | Not reproducible | Assume 70/15/15, `random_state=42`, sklearn default L2; all in `src/config.py` |


In [11]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent / 'src'))
import pandas as pd
pd.set_option('display.max_columns', 60)
pd.set_option('display.width', 170)
import experiment
out = experiment.run_experiment(verbose=False)
pd.DataFrame([out['summary']['split_sizes']])

,train,val,test,train_pos_rate,test_pos_rate
0,1028,221,221,0.1605,0.1629


The 1-max-out search is cached under `outputs/cache/maxout_*.json` (keyed by a signature of the model config and protocol) so it is not repeated on every run.

### 2.2 Correction A - split first, then everything else

`data.three_way_split` does a stratified 70/15/15 split on the raw frame. `preprocessing.build_preprocessor` builds a `ColumnTransformer` (StandardScaler + OneHotEncoder keeping all levels, matching the paper's dummy coding) fit on training rows only. Validation and test are transformed with those frozen statistics; the test set is scored exactly once.

### 2.3 Feature selection - max-out, leakage-safe

`maxout.maxout_select` reimplements 1-max-out: greedily drop the single feature whose removal most improves validation F1, stop when no removal helps. Repeated over 7 random inner train/val splits; a feature is eliminated if dropped in more than 4 runs. Each run re-fits the scaler/encoder on its own inner-training rows.

In [12]:
for model, elim in out['summary']['maxout_eliminated'].items():
    print(f'{model}: {len(elim)} eliminated')
    for f in elim:
        print('   -', f)

LogisticRegression: 8 eliminated
   - Department_Research & Development
   - Education
   - EducationField_Human Resources
   - EducationField_Marketing
   - Gender_Female
   - JobLevel
   - MonthlyRate
   - PerformanceRating
RandomForest: 2 eliminated
   - DailyRate
   - YearsAtCompany


In [13]:
pd.Series(out['summary']['maxout_drop_counts']['LogisticRegression'], name='times_dropped_over_7_runs').head(15)

Gender_Female                        6
JobLevel                             6
PerformanceRating                    6
Department_Research & Development    6
EducationField_Human Resources       5
Education                            5
MonthlyRate                          5
EducationField_Marketing             5
JobRole_Research Scientist           4
Department_Human Resources           4
Department_Sales                     4
EducationField_Life Sciences         4
Over18_Y                             4
JobRole_Healthcare Representative    4
HourlyRate                           4
Name: times_dropped_over_7_runs, dtype: int64

**Comparison with the paper.** The paper eliminated 21 dummy features; our leakage-safe LR run is more conservative (8). Reassuringly, **every** feature our LR run eliminates - `JobLevel`, `MonthlyRate`, `PerformanceRating`, `Gender_Female`, `Education`, `Department_Research & Development`, `EducationField_Human Resources`, `EducationField_Marketing` - also appears in the paper's omitted list, and they are precisely the collinear / low-signal dummies the EDA flagged. Random Forest eliminates a different, smaller set (`DailyRate`, `YearsAtCompany`) because it uses a different wrapper metric - which is the point of running the search per model.

### 2.4 Correction B - a fair model comparison

One split, one seed, one preprocessor. Both models get `class_weight='balanced'`. Each is evaluated with all features and with its own max-out selection, at a decision threshold chosen on the validation set. 95% CIs on test F1 are 2000-resample percentile bootstraps of the test set.

In [14]:
cmp = out['comparison']
cmp[['model','variant','n_features','val_threshold','f1_at_0.5','accuracy','precision','recall','f1','f1_ci','roc_auc','pr_auc']]

,model,variant,n_features,val_threshold,f1_at_0.5,accuracy,precision,recall,f1,f1_ci,roc_auc,pr_auc
0,LogReg (paper) + max-out,paper,NaN,NaN,NaN,0.8100,0.4300,0.8200,0.5600,NaN,NaN,NaN
1,"LogReg (paper), all features",paper,NaN,NaN,NaN,0.7800,0.3900,0.8200,0.5300,NaN,NaN,NaN
2,LogisticRegression,all_features,52.0,0.7156,0.5143,0.8235,0.4571,0.4444,0.4507,"(0.3, 0.5834)",0.8350,0.6026
3,LogisticRegression,maxout,44.0,0.7337,0.5094,0.8326,0.4848,0.4444,0.4638,"(0.3103, 0.6)",0.8333,0.6035
4,RandomForest,all_features,52.0,0.2367,0.1818,0.8100,0.4286,0.5000,0.4615,"(0.3125, 0.5898)",0.8047,0.4886
5,RandomForest,maxout,50.0,0.3233,0.1860,0.8507,0.5517,0.4444,0.4923,"(0.3333, 0.6316)",0.8282,0.5134


![fair comparison](../outputs/figures/06_fair_model_comparison.png)

### 2.5 Results vs the paper -- this is the Assignment #3 baseline (E0)

| Model | Paper | Baseline (threshold tuned on validation) |
|---|---|---|
| LR + max-out | acc 0.81, P 0.43, R 0.82, **F1 0.56** | acc 0.83, P 0.48, R 0.44, **F1 0.46**  (95% CI 0.31-0.60) |
| LR, all features | acc 0.78, P 0.39, R 0.82, **F1 0.53** | acc 0.82, P 0.46, R 0.44, **F1 0.45**  (95% CI 0.30-0.58) |
| RF + max-out | not run on a comparable basis | acc 0.85, P 0.55, R 0.44, **F1 0.49**  (95% CI 0.33-0.63) |
| RF, all features | not run on a comparable basis | acc 0.81, P 0.43, R 0.50, **F1 0.46**  (95% CI 0.31-0.59) |

**This LR + max-out row (acc 0.83 / P 0.48 / R 0.44 / F1 0.46) is the Assignment #3 baseline (E0)** that every later experiment (E1, E2, ... ) is compared against, using the same preprocessing, features, model configuration, split, and evaluation procedure.

### Discussion

**Could we reproduce the reported performance?** Not the headline numbers. Our LR + max-out reaches F1 ~ 0.46 against the paper's 0.56, and recall 0.44 against 0.82. The paper's combination of recall 0.82 with precision 0.43 is the profile of a model run at a **very low decision threshold** (recall bought at the cost of precision). At a fixed 0.5 cut our LR actually scores higher (`f1_at_0.5` ~ 0.51); maximising *validation* F1 moves the threshold the other way, trading recall for precision. Forcing recall ~ 0.82 on our split drives precision below 0.30. The most likely explanations for the gap: the paper never fixes a threshold, regularisation strength, or split seed, and its Section 5.3 wording suggests some steps touched the whole dataset.

**Did max-out actually help?** Marginally, and not distinguishably. LR: F1 0.45 -> 0.46; RF: 0.46 -> 0.49. The bootstrap CIs overlap almost completely in both cases, so the improvement the paper attributes to max-out (F1 0.53 -> 0.56) **cannot be called statistically significant** in a leakage-safe setting - consistent with how small the paper's own delta is. What max-out does deliver is a smaller model (44 vs 52 features for LR) at no measurable cost.

**LR + max-out vs RF + max-out** - the comparison the paper should have made. At the tuned operating point RF edges LR on F1 (0.49 vs 0.46), but the CIs (0.33-0.63 vs 0.31-0.60) overlap heavily. The threshold-free view is clearer and favours **LR**: PR-AUC 0.60 vs 0.51, ROC-AUC 0.83 vs 0.83. LR ranks positives better across all thresholds; RF only pulls ahead at one tuned point. Net: no strong evidence either model dominates, with a mild edge to LR as the more robust ranker - consistent with keeping LR as the main model for Assignment #3 (per the brief, model capacity is not what is being improved).

**Did the methodological corrections change the outcome?** Yes. (1) The leakage-safe split plus a single honest test evaluation drops F1 from the paper's 0.56 to ~0.46. (2) The fixed-0.5-cut comparison is actively misleading here: at 0.5, RF + max-out scores F1 0.19 (`f1_at_0.5`) purely because `class_weight` never moved its vote threshold - tuning it on validation lifts the same model to 0.49. Any imbalanced-data comparison at a fixed 0.5 is unreliable.

**Limitations (carried into Assignment #3).** A single 70/15/15 split for this baseline table (no repeated CV for the headline number here -- Assignment #3 introduces stratified k-fold CV for all later experiments); the 221-row test set makes every CI wide (~0.30 F1 span); the paper's unspecified hyperparameters force assumptions; `class_weight='balanced'` and the F1-optimal threshold are our choices, applied equally to both models; the ordinal-as-numeric encoding is inherited from the paper and is a candidate for improvement.

### 2.6 Post-processing - coefficient-confidence analysis (Section 5.3)

300 bootstrap resamples of the **training** set, LR + max-out refit on each, reporting the mean, SD and coefficient of variation (SD / |mean|) of every coefficient. Low CV = stable estimate. The estimator bootstrapped here is the same object used for the headline result.

In [15]:
out['bootstrap_coef']

,feature,mean,std,coef_of_variation
0,OverTime_No,-0.965069,0.137966,0.142960
1,OverTime_Yes,0.941051,0.138223,0.146882
2,JobSatisfaction,-0.529101,0.119703,0.226239
3,BusinessTravel_Non-Travel,-1.215292,0.285903,0.235254
4,BusinessTravel_Travel_Frequently,0.886810,0.215861,0.243413
5,EnvironmentSatisfaction,-0.448695,0.115947,0.258410
6,DistanceFromHome,0.402783,0.115227,0.286077
7,NumCompaniesWorked,0.476738,0.140561,0.294839
8,JobRole_Laboratory Technician,1.041252,0.314892,0.302416
9,TrainingTimesLastYear,-0.367136,0.112632,0.306786


In [16]:
out['final_coef'].head(12)   # single-fit coefficients, largest magnitude first

,feature,coefficient
0,JobRole_Research Director,-1.285884
1,BusinessTravel_Non-Travel,-1.137870
2,JobRole_Laboratory Technician,1.001533
3,OverTime_No,-0.870938
4,OverTime_Yes,0.843172
5,BusinessTravel_Travel_Frequently,0.803396
6,YearsAtCompany,0.795041
7,JobRole_Sales Representative,0.756450
8,YearsWithCurrManager,-0.651394
9,Department_Sales,0.590970


![bootstrap coefficients](../outputs/figures/07_bootstrap_coefficients.png)

**Reading.** The largest, most stable coefficients (CV < 0.5) are `OverTime`, `BusinessTravel`, `JobSatisfaction`, `EnvironmentSatisfaction`, `JobRole_Laboratory Technician`, `MaritalStatus_Single` - and they line up with the EDA's feature-target section, so these are the signals to trust. The unstable ones (CV >= 0.5) - `TotalWorkingYears`, `Department_Sales`, `JobRole_Sales Representative`, `Age` - sit inside the high-VIF / collinear blocks the EDA identified in section 1.4. This is direct evidence for the caution raised there: the paper's practice of reading individual coefficients as feature importance is fragile for exactly those features, and the bootstrap makes the fragility visible.

## Next steps (outside this notebook)

With E0 established above, later Assignment #3 notebooks/scripts will: (1) formulate hypotheses from the EDA findings in Part 1, (2) evaluate preprocessing improvements one at a time under shared stratified k-fold CV, (3) tune the main model's hyperparameters under the same CV protocol without touching the test set, and (4) score the final pipeline against this baseline on the held-out test set.